<a href="https://colab.research.google.com/github/Ethan06lu/campus-coffee/blob/main/campus_coffee.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Game Of The Week

This will be about our campus coffee

In [7]:
from google.colab import drive
import pandas as pd

drive.mount('/content/drive')
games = pd.read_csv("/content/drive/MyDrive/steam_games (1).csv")
print(f"✅ Loaded {len(games)} games")

ValueError: mount failed

In [8]:
def find_col(df, candidates):
    """Return the first column whose lowercase name matches/contains a candidate."""
    lower = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand in lower:
            return lower[cand]
    for cand in candidates:
        for low, orig in lower.items():
            if cand in low:
                return orig
    raise KeyError(f"No column found for {candidates}. Columns: {list(df.columns)}")

In [9]:
def top_rated_by_players(games, n=10, min_reviews=100):
    """Top games by player rating. Uses positive/negative reviews if available,
    otherwise falls back to a rating column. min_reviews filters out games
    with too few reviews to be trustworthy."""
    name = find_col(games, ["name", "title"])
    df = games.copy()
    try:
        pos = find_col(df, ["positive_ratings", "positive", "positive_reviews"])
        neg = find_col(df, ["negative_ratings", "negative", "negative_reviews"])
        df["total_reviews"] = df[pos] + df[neg]
        df["player_rating_pct"] = (df[pos] / df["total_reviews"] * 100).round(1)
        df = df[df["total_reviews"] >= min_reviews]
        cols = [name, "player_rating_pct", "total_reviews"]
    except KeyError:
        rating = find_col(df, ["rating", "user_score", "review_score", "score"])
        df["player_rating_pct"] = df[rating]
        cols = [name, "player_rating_pct"]
    return df.sort_values("player_rating_pct", ascending=False).head(n)[cols].reset_index(drop=True)



In [10]:
def critic_view(games, top_games=None, n=10):
    """Show critic scores/reviews for the top player-rated games."""
    name = find_col(games, ["name", "title"])
    if top_games is None:
        top_games = top_rated_by_players(games, n=n)
    subset = games[games[name].isin(top_games[name])]

    # Pull any critic-related columns that exist
    critic_cols = [c for c in games.columns
                   if any(k in c.lower() for k in
                          ["metacritic", "critic", "review_text", "summary", "reviews_text"])]
    if not critic_cols:
        print("No critic columns found. Columns available:", list(games.columns))
        return subset[[name]]

    result = subset[[name] + critic_cols]
    # Preserve the player-rank order
    result = result.set_index(name).loc[top_games[name].tolist()].reset_index()
    return result

